## Step 1: Imports & Configuration

In [ ]:
import os, re, random, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance
from tqdm import tqdm
from scipy.stats import mannwhitneyu
from scipy import ndimage
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed(SEED)

BASE_DIR   = os.path.expanduser('~/LUMINOUS_Database')
IMAGE_DIR  = os.path.join(BASE_DIR, 'B-mode')
MASK_DIR   = os.path.join(BASE_DIR, 'Masks')
OUTPUT_DIR = os.path.expanduser('~/unet_improved_final')
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE      = 256
BATCH_SIZE    = 8
LEARNING_RATE = 1e-4
NUM_EPOCHS    = 50
PATIENCE      = 10
# LUMINOUS: 820x614px, 12cm wide, 6cm long
PX_TO_CM_H  = 6.0  / 614.0
PX_TO_CM_W  = 12.0 / 820.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('='*60)
print(f'  Device  : {device}')
if torch.cuda.is_available():
    print(f'  GPU     : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'  Images  : {len([f for f in os.listdir(IMAGE_DIR) if f.endswith(".tif")])}')
print(f'  Masks   : {len(os.listdir(MASK_DIR))}')
print('='*60)

  Device  : cuda
  GPU     : Tesla V100-PCIE-32GB
  VRAM    : 34.1 GB
  Images  : 341
  Masks   : 386


## Step 2: Build Dataset with Verified Midpoint Crop Strategy


In [ ]:
def get_mask_side_single(mask_path):
    """For single masks: determine side from centroid position."""
    m = np.array(Image.open(mask_path).convert('L'))
    cols = np.where(m > 127)[1]
    if len(cols) == 0: return 'Unknown'
    cx = cols.mean(); W = m.shape[1]
    if cx < W * 0.45: return 'Left'
    elif cx > W * 0.55: return 'Right'
    else: return 'Center'


image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith('.tif')])
mask_files  = sorted([f for f in os.listdir(MASK_DIR)  if f.endswith('.tif')])

data_pairs = []
for img_file in image_files:
    m = re.match(r'^(\d+)_(\d+)_', img_file)
    if not m: continue
    sid, vid = m.group(1), m.group(2)
    masks = sorted([f for f in mask_files if f.startswith(f'{sid}_{vid}_')])

    for mf in masks:
        mask_path = os.path.join(MASK_DIR, mf)
        img_path  = os.path.join(IMAGE_DIR, img_file)

        if 'Mask1' in mf:
            # RIGHT LM — crop to RIGHT HALF of image (verified for all 45 pairs)
            side      = 'Right'
            crop_type = 'right_half'
        elif 'Mask2' in mf:
            # LEFT LM — crop to LEFT HALF of image (verified for all 45 pairs)
            side      = 'Left'
            crop_type = 'left_half'
        else:
            # Single mask — use full image, no crop needed
            side      = get_mask_side_single(mask_path)
            crop_type = 'full'

        data_pairs.append({
            'img_path'  : img_path,
            'mask_path' : mask_path,
            'subject_id': sid,
            'visit_id'  : vid,
            'visit_int' : int(vid),
            'side'      : side,
            'crop_type' : crop_type,   # 'full' | 'left_half' | 'right_half'
            'mask_type' : 'dual' if crop_type != 'full' else 'single',
            'base'      : f'{sid}_{vid}',
        })

sides = Counter(p['side'] for p in data_pairs)
crops = Counter(p['crop_type'] for p in data_pairs)
print(f'Total pairs     : {len(data_pairs)}')
print(f'  Left LM       : {sides["Left"]}')
print(f'  Right LM      : {sides["Right"]}')
print(f'  Center LM     : {sides["Center"]}')
print(f'  Full image    : {crops["full"]}  (single mask — no crop)')
print(f'  Left half crop: {crops["left_half"]}  (Mask2 — LEFT LM)')
print(f'  Right half crop:{crops["right_half"]}  (Mask1 — RIGHT LM)')
print(f'  Subjects      : {len(set(p["subject_id"] for p in data_pairs))}')

Total pairs     : 386
  Left LM       : 195
  Right LM      : 51
  Center LM     : 140
  Full image    : 296  (single mask — no crop)
  Left half crop: 45  (Mask2 — LEFT LM)
  Right half crop:45  (Mask1 — RIGHT LM)
  Subjects      : 109


## Step 3: Dataset Class with Midpoint Crop + Augmentation

In [3]:
def apply_crop(img_pil, mask_pil, crop_type):
    """
    Apply midpoint crop based on which LM muscle we want.
    Verified: W=820, midpoint=410, gap between muscles 20-73px.
    full       -> no crop (single mask images)
    left_half  -> crop x=0:410   (LEFT LM = Mask2)
    right_half -> crop x=410:820 (RIGHT LM = Mask1)
    """
    if crop_type == 'full':
        return img_pil, mask_pil
    W, H = img_pil.size  # PIL: (width, height)
    mid  = W // 2
    if crop_type == 'left_half':
        box = (0, 0, mid, H)     # (left, top, right, bottom)
    else:  # right_half
        box = (mid, 0, W, H)
    return img_pil.crop(box), mask_pil.crop(box)


class LuminousDataset(Dataset):
    def __init__(self, pairs, size=IMG_SIZE, augment=False):
        self.pairs   = pairs
        self.size    = size
        self.augment = augment

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p    = self.pairs[idx]
        img  = Image.open(p['img_path']).convert('L')
        mask = Image.open(p['mask_path']).convert('L')

        # Apply midpoint crop for dual-mask images
        img, mask = apply_crop(img, mask, p['crop_type'])

        img  = img.resize((self.size, self.size), Image.BILINEAR)
        mask = mask.resize((self.size, self.size), Image.NEAREST)

        if self.augment:
            if random.random() > 0.5:
                img  = img.transpose(Image.FLIP_LEFT_RIGHT)
                mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() > 0.5:
                img  = img.transpose(Image.FLIP_TOP_BOTTOM)
                mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
            ang  = random.uniform(-15, 15)
            img  = img.rotate(ang, resample=Image.BILINEAR)
            mask = mask.rotate(ang, resample=Image.NEAREST)
            if random.random() > 0.5:
                img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
            if random.random() > 0.5:
                img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))

        it = torch.from_numpy(np.array(img)).float().unsqueeze(0) / 255.0
        mt = torch.from_numpy(np.array(mask)).float().unsqueeze(0) / 255.0
        return it, (mt > 0.5).float()


random.shuffle(data_pairs)
sp              = int(0.8 * len(data_pairs))
train_pairs     = data_pairs[:sp]
val_pairs_split = data_pairs[sp:]

train_ds = LuminousDataset(train_pairs,     augment=True)
val_ds   = LuminousDataset(val_pairs_split, augment=False)
full_ds  = LuminousDataset(data_pairs,      augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
full_loader  = DataLoader(full_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)}  |  Val: {len(val_ds)}')

# Visualize samples with crop applied
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
sample_pool = [p for p in data_pairs if p['crop_type']=='left_half'][:3] + \
              [p for p in data_pairs if p['crop_type']=='right_half'][:3] + \
              [p for p in data_pairs if p['crop_type']=='full'][:3]
for ax, p in zip(axes.flatten(), sample_pool):
    img  = Image.open(p['img_path']).convert('L')
    mask = Image.open(p['mask_path']).convert('L')
    img, mask = apply_crop(img, mask, p['crop_type'])
    img_np  = np.array(img)
    mask_np = np.array(mask)
    rgb     = np.stack([img_np]*3, axis=-1).copy()
    rgb[mask_np>127, 1] = np.clip(rgb[mask_np>127, 1].astype(int) + 80, 0, 255)
    ax.imshow(rgb)
    ax.set_title(f"S{p['subject_id']} V{p['visit_id']}\n{p['side']} | {p['crop_type']}", fontsize=9)
    ax.axis('off')
plt.suptitle('Dataset Samples After Crop (Green=LM Mask)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'sample_images.png'), dpi=100); plt.close()
print('Saved: sample_images.png')

Train: 308  |  Val: 78
Saved: sample_images.png


## Step 4: U-Net Architecture

In [4]:
class DoubleConv(nn.Module):
    def __init__(self, a, b, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(a,b,3,1,1,bias=False), nn.BatchNorm2d(b), nn.ReLU(True),
            nn.Dropout2d(drop),
            nn.Conv2d(b,b,3,1,1,bias=False), nn.BatchNorm2d(b), nn.ReLU(True))
    def forward(self, x): return self.net(x)

class UNET(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, feats=[64,128,256,512]):
        super().__init__()
        self.downs=nn.ModuleList(); self.ups=nn.ModuleList(); self.pool=nn.MaxPool2d(2,2)
        for f in feats: self.downs.append(DoubleConv(in_ch,f)); in_ch=f
        self.bottleneck=DoubleConv(feats[-1],feats[-1]*2,drop=0.2)
        for f in reversed(feats):
            self.ups.append(nn.ConvTranspose2d(f*2,f,2,2))
            self.ups.append(DoubleConv(f*2,f))
        self.final=nn.Conv2d(feats[0],out_ch,1)
    def forward(self,x):
        skips=[]
        for d in self.downs: x=d(x); skips.append(x); x=self.pool(x)
        x=self.bottleneck(x)
        for i in range(0,len(self.ups),2):
            x=self.ups[i](x); s=skips[-(i//2+1)]
            if x.shape!=s.shape: x=F.interpolate(x,size=s.shape[2:])
            x=torch.cat([s,x],1); x=self.ups[i+1](x)
        return self.final(x)

model=UNET().to(device)
print(f'U-Net params: {sum(p.numel() for p in model.parameters()):,}')

U-Net params: 31,036,481


## Step 5: Combined BCE + Dice Loss

In [5]:
class DiceLoss(nn.Module):
    def forward(self,lo,ta,sm=1.0):
        p=torch.sigmoid(lo); i=(p*ta).sum(dim=(2,3))
        return 1-((2*i+sm)/(p.sum(dim=(2,3))+ta.sum(dim=(2,3))+sm)).mean()

class CombLoss(nn.Module):
    def __init__(self,w=0.5): super().__init__(); self.w=w; self.bce=nn.BCEWithLogitsLoss(); self.dice=DiceLoss()
    def forward(self,lo,ta): return self.w*self.bce(lo,ta)+(1-self.w)*self.dice(lo,ta)

loss_fn   = CombLoss(0.5)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer,'min',patience=5,factor=0.5,min_lr=1e-6)
scaler    = torch.cuda.amp.GradScaler()
print('Loss: 0.5*BCE + 0.5*Dice | Adam | ReduceLROnPlateau')

Loss: 0.5*BCE + 0.5*Dice | Adam | ReduceLROnPlateau


## Step 6: Training with Early Stopping

In [6]:
def dice_batch(lo,ta,th=0.5,sm=1e-6):
    p=(torch.sigmoid(lo)>th).float(); i=(p*ta).sum(dim=(2,3))
    return ((2*i+sm)/(p.sum(dim=(2,3))+ta.sum(dim=(2,3))+sm)).mean().item()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    tl,td=0.0,0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs,masks in loader:
            imgs,masks=imgs.to(device),masks.to(device)
            with torch.cuda.amp.autocast(): preds=model(imgs); loss=loss_fn(preds,masks)
            if train: optimizer.zero_grad(); scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            tl+=loss.item(); td+=dice_batch(preds.detach() if train else preds,masks)
    return tl/len(loader),td/len(loader)

MODEL_PATH=os.path.join(OUTPUT_DIR,'unet_best.pth')
trl,vll,trd,vld=[],[],[],[]; best_v=float('inf'); ni=0
print(f'Training up to {NUM_EPOCHS} epochs (early stop patience={PATIENCE})')
print(f'{"Ep":>4} {"TrLoss":>8} {"TrDice":>8} {"VaLoss":>8} {"VaDice":>8} {"LR":>10}')
print('-'*55)
for ep in range(1,NUM_EPOCHS+1):
    tl,td=run_epoch(train_loader,True); vl,vd=run_epoch(val_loader,False)
    trl.append(tl);vll.append(vl);trd.append(td);vld.append(vd)
    scheduler.step(vl); lr=optimizer.param_groups[0]['lr']
    print(f'{ep:>4} {tl:>8.4f} {td:>8.4f} {vl:>8.4f} {vd:>8.4f} {lr:>10.2e}')
    if vl<best_v: best_v=vl; ni=0; torch.save(model.state_dict(),MODEL_PATH); print(f'     *** Saved (dice={vd:.4f}) ***')
    else:
        ni+=1
        if ni>=PATIENCE: print(f'Early stop at epoch {ep}'); break
n_ep=len(trl)
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].plot(range(1,n_ep+1),trl,'b-o',ms=3,label='Train'); axes[0].plot(range(1,n_ep+1),vll,'r-o',ms=3,label='Val')
axes[0].set_title('Combined Loss'); axes[0].legend(); axes[0].grid(True)
axes[1].plot(range(1,n_ep+1),trd,'b-o',ms=3,label='Train'); axes[1].plot(range(1,n_ep+1),vld,'r-o',ms=3,label='Val')
axes[1].set_title('Dice Score'); axes[1].legend(); axes[1].grid(True)
plt.suptitle('U-Net Training — Midpoint-Crop Strategy',fontsize=13)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'training_curves.png'),dpi=100); plt.close()
print('Saved: training_curves.png')

Training up to 50 epochs (early stop patience=10)
  Ep   TrLoss   TrDice   VaLoss   VaDice         LR
-------------------------------------------------------
   1   0.6929   0.4109   0.6716   0.0000   1.00e-04
     *** Saved (dice=0.0000) ***
   2   0.5540   0.6765   0.5144   0.6841   1.00e-04
     *** Saved (dice=0.6841) ***
   3   0.4998   0.7437   0.5377   0.6503   1.00e-04
   4   0.4777   0.7657   0.4534   0.7913   1.00e-04
     *** Saved (dice=0.7913) ***
   5   0.4573   0.7744   0.4609   0.7517   1.00e-04
   6   0.4389   0.7900   0.4208   0.8068   1.00e-04
     *** Saved (dice=0.8068) ***
   7   0.4176   0.8160   0.3998   0.8370   1.00e-04
     *** Saved (dice=0.8370) ***
   8   0.4068   0.8053   0.3955   0.8019   1.00e-04
     *** Saved (dice=0.8019) ***
   9   0.3883   0.8199   0.3809   0.8156   1.00e-04
     *** Saved (dice=0.8156) ***
  10   0.3765   0.8212   0.3612   0.8171   1.00e-04
     *** Saved (dice=0.8171) ***
  11   0.3637   0.8202   0.3643   0.7919   1.00e-04
  12  

## Step 7: Evaluation — Overall + Per Side + Per Type

In [7]:
model.load_state_dict(torch.load(MODEL_PATH,map_location=device)); model.eval()
all_dice,all_iou,all_prec,all_rec=[],[],[],[]
side_dice={'Left':[],'Right':[],'Center':[]}
type_dice={'single':[],'dual':[]}
sm=1e-6

with torch.no_grad():
    for bi in tqdm(range(0,len(val_pairs_split),BATCH_SIZE),'Eval'):
        batch=val_pairs_split[bi:bi+BATCH_SIZE]
        imgs_l,masks_l=[],[]
        for p in batch:
            img  = Image.open(p['img_path']).convert('L')
            mask = Image.open(p['mask_path']).convert('L')
            img, mask = apply_crop(img, mask, p['crop_type'])
            imgs_l.append(torch.from_numpy(np.array(img.resize((IMG_SIZE,IMG_SIZE),Image.BILINEAR))).float().unsqueeze(0)/255.0)
            masks_l.append((torch.from_numpy(np.array(mask.resize((IMG_SIZE,IMG_SIZE),Image.NEAREST))).float().unsqueeze(0)/255.0>0.5).float())
        imgs_t=torch.stack(imgs_l).to(device); masks_t=torch.stack(masks_l).to(device)
        preds=(torch.sigmoid(model(imgs_t))>0.5).float()
        for i,p in enumerate(batch):
            pp=preds[i].flatten(); tt=masks_t[i].flatten()
            tp=(pp*tt).sum().item(); fp=(pp*(1-tt)).sum().item(); fn=((1-pp)*tt).sum().item()
            d=(2*tp+sm)/(2*tp+fp+fn+sm)
            all_dice.append(d); all_iou.append((tp+sm)/(tp+fp+fn+sm))
            all_prec.append((tp+sm)/(tp+fp+sm)); all_rec.append((tp+sm)/(tp+fn+sm))
            if p['side'] in side_dice: side_dice[p['side']].append(d)
            type_dice[p['mask_type']].append(d)

md=np.mean(all_dice); mi=np.mean(all_iou); mp=np.mean(all_prec); mr=np.mean(all_rec)
mf=2*mp*mr/(mp+mr+1e-6)
print('='*55)
print('  SEGMENTATION RESULTS (Midpoint-Crop)')
print('='*55)
print(f'  Dice      : {md:.4f}')
print(f'  IoU       : {mi:.4f}')
print(f'  Precision : {mp:.4f}')
print(f'  Recall    : {mr:.4f}')
print(f'  F1        : {mf:.4f}')
print()
print('  Per-Side Dice:')
for s,dices in side_dice.items():
    if dices: print(f'    {s:6s}: {np.mean(dices):.4f} ± {np.std(dices):.4f}  (n={len(dices)})')
print('  Per-Type Dice:')
for t,dices in type_dice.items():
    if dices: print(f'    {t:6s}: {np.mean(dices):.4f}  (n={len(dices)})')
print('='*55)

fig,axes=plt.subplots(1,4,figsize=(18,4))
for ax,sc,nm,cl in zip(axes,[all_dice,all_iou,all_prec,all_rec],
    ['Dice','IoU','Precision','Recall'],['steelblue','teal','coral','mediumpurple']):
    ax.hist(sc,bins=20,color=cl,edgecolor='k',alpha=0.8)
    ax.axvline(np.mean(sc),color='red',linestyle='--',label=f'Mean={np.mean(sc):.3f}'); ax.set_title(nm); ax.legend()
plt.suptitle('Segmentation Performance — Midpoint-Crop Strategy',fontsize=13)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'segmentation_scores.png'),dpi=100); plt.close()
print('Saved: segmentation_scores.png')

Eval: 100%|██████████| 10/10 [00:00<00:00, 12.42it/s]


  SEGMENTATION RESULTS (Midpoint-Crop)
  Dice      : 0.9120
  IoU       : 0.8434
  Precision : 0.9219
  Recall    : 0.9117
  F1        : 0.9168

  Per-Side Dice:
    Left  : 0.9138 ± 0.0657  (n=39)
    Right : 0.9100 ± 0.0852  (n=11)
    Center: 0.9102 ± 0.0442  (n=28)
  Per-Type Dice:
    single: 0.9130  (n=55)
    dual  : 0.9097  (n=23)
Saved: segmentation_scores.png


## Step 8: Visualize Predictions (Left/Right/Center/Single/Dual)

In [8]:
model.eval()
sample_pool = (
    [p for p in val_pairs_split if p['crop_type']=='left_half'][:2] +
    [p for p in val_pairs_split if p['crop_type']=='right_half'][:2] +
    [p for p in val_pairs_split if p['crop_type']=='full'][:1]
)
if not sample_pool: sample_pool = val_pairs_split[:5]

fig,axes=plt.subplots(len(sample_pool),4,figsize=(16,4*len(sample_pool)))
if len(sample_pool)==1: axes=[axes]
for row,p in enumerate(sample_pool):
    img  = Image.open(p['img_path']).convert('L')
    mask = Image.open(p['mask_path']).convert('L')
    img, mask = apply_crop(img, mask, p['crop_type'])
    img  = img.resize((IMG_SIZE,IMG_SIZE),Image.BILINEAR)
    mask = mask.resize((IMG_SIZE,IMG_SIZE),Image.NEAREST)
    img_np  = np.array(img)/255.0
    mask_np = (np.array(mask)>127).astype(np.float32)
    it=torch.from_numpy(img_np).float().unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad(): prob=torch.sigmoid(model(it)).squeeze().cpu().numpy()
    pred=(prob>0.5).astype(np.float32)
    tp=pred*mask_np; fp=pred*(1-mask_np); fn=(1-pred)*mask_np
    d=(2*tp.sum()+1e-6)/(pred.sum()+mask_np.sum()+1e-6)
    ov=np.stack([img_np]*3,axis=-1)
    ov[...,1]=np.clip(ov[...,1]+0.4*tp,0,1)
    ov[...,0]=np.clip(ov[...,0]+0.4*fp,0,1)
    ov[...,2]=np.clip(ov[...,2]+0.4*fn,0,1)
    axes[row][0].imshow(img_np,cmap='gray')
    axes[row][0].set_title(f"S{p['subject_id']} V{p['visit_id']}\n{p['side']} | {p['crop_type']}")
    axes[row][0].axis('off')
    axes[row][1].imshow(mask_np,cmap='gray'); axes[row][1].set_title('Ground Truth'); axes[row][1].axis('off')
    axes[row][2].imshow(pred,cmap='gray'); axes[row][2].set_title(f'Predicted (Dice={d:.3f})'); axes[row][2].axis('off')
    axes[row][3].imshow(ov); axes[row][3].set_title('TP=G FP=R FN=B'); axes[row][3].axis('off')
plt.suptitle('U-Net Predictions — Midpoint-Crop Strategy (Fixed)',fontsize=14)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'predictions.png'),dpi=100,bbox_inches='tight'); plt.close()
print('Saved: predictions.png')

Saved: predictions.png


## Step 9: EI + CSA Extraction with Correct Scaling

In [9]:
model.eval(); records=[]
with torch.no_grad():
    for bi in tqdm(range(0,len(data_pairs),BATCH_SIZE),'Extracting EI+CSA'):
        batch=data_pairs[bi:bi+BATCH_SIZE]
        imgs_l=[]
        crop_dims=[]  # store crop dimensions for correct CSA scaling
        for p in batch:
            img  = Image.open(p['img_path']).convert('L')
            mask = Image.open(p['mask_path']).convert('L')
            img, mask = apply_crop(img, mask, p['crop_type'])
            orig_w, orig_h = img.size  # after crop
            crop_dims.append((orig_h, orig_w))
            imgs_l.append(torch.from_numpy(np.array(img.resize((IMG_SIZE,IMG_SIZE),Image.BILINEAR))).float().unsqueeze(0)/255.0)
        imgs_t=torch.stack(imgs_l).to(device)
        preds=(torch.sigmoid(model(imgs_t))>0.5).float()
        for i,p in enumerate(batch):
            it=imgs_t[i].squeeze(); mt=preds[i].squeeze(); ms=mt.sum().item()
            orig_h, orig_w = crop_dims[i]
            # Scale pixels back to original crop size for CSA
            px_h = orig_h / IMG_SIZE; px_w = orig_w / IMG_SIZE
            cm_per_px_h = 6.0  / 614.0
            cm_per_px_w = 12.0 / 820.0
            if ms>0:
                rx  = it[mt>0.5].cpu().numpy()
                ei  = float(rx.mean()); eis = float(rx.std())
                csa = float(ms * px_h*cm_per_px_h * px_w*cm_per_px_w)
                mr  = ms/mt.numel()
            else:
                rx  = it.cpu().numpy().flatten()
                ei  = float(rx.mean()); eis = float(rx.std()); csa=0.0; mr=0.0
            records.append({'subject_id':p['subject_id'],'visit_id':p['visit_id'],
                'visit_int':p['visit_int'],'side':p['side'],
                'crop_type':p['crop_type'],'mask_type':p['mask_type'],
                'base':p['base'],'ei_raw':ei,'ei_std':eis,
                'csa_cm2':csa,'mask_ratio':float(mr)})

df=pd.DataFrame(records)
print(f'Extracted: {len(df)} | Left:{len(df[df["side"]=="Left"])} Right:{len(df[df["side"]=="Right"])} Center:{len(df[df["side"]=="Center"])}')
print(f'Mean EI={df["ei_raw"].mean():.4f}  Mean CSA={df["csa_cm2"].mean():.2f}cm²')

Extracting EI+CSA: 100%|██████████| 49/49 [00:03<00:00, 12.97it/s]

Extracted: 386 | Left:195 Right:51 Center:140
Mean EI=0.2252  Mean CSA=4.65cm²


## Step 10: Standardization — 4 Methods

In [10]:
ei=df['ei_raw'].values
df['ei_zscore'] =(ei-ei.mean())/(ei.std()+1e-8)
df['ei_minmax'] =(ei-ei.min())/(ei.max()-ei.min()+1e-8)
df['ei_phantom']=(ei/ei.mean())*100
q25,q75=np.percentile(ei,25),np.percentile(ei,75)
df['ei_robust'] =(ei-np.median(ei))/(q75-q25+1e-8)
cv_raw    =(ei.std()/ei.mean())*100
cv_phantom=(df['ei_phantom'].std()/df['ei_phantom'].mean())*100
print(f'CV Raw: {cv_raw:.2f}%  Phantom: {cv_phantom:.2f}%')
print(f'Z-score: mean={df["ei_zscore"].mean():.3f} std={df["ei_zscore"].std():.3f}')

fig,axes=plt.subplots(2,2,figsize=(14,10)); axes=axes.flatten()
for ax,(title,col,color) in zip(axes,[
    ('Raw EI (normalized pixel)','ei_raw','steelblue'),
    ('Z-score Standardized','ei_zscore','teal'),
    ('Min-Max [0,1]','ei_minmax','coral'),
    ('Phantom-Reference (% of mean)','ei_phantom','mediumpurple')]):
    v=df[col].values
    ax.hist(v,bins=30,color=color,edgecolor='k',alpha=0.8)
    ax.axvline(v.mean(),color='red',linestyle='--',label=f'Mean={v.mean():.3f}\nStd={v.std():.3f}')
    ax.set_title(title,fontsize=11); ax.legend(); ax.set_xlabel('Value'); ax.set_ylabel('Count')
plt.suptitle('EI Standardization Methods — LUMINOUS LM Muscle (Fixed)',fontsize=14)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'standardization_comparison.png'),dpi=100); plt.close()
print('Saved: standardization_comparison.png')

CV Raw: 29.66%  Phantom: 29.70%
Z-score: mean=-0.000 std=1.001
Saved: standardization_comparison.png


## Step 11: Heckmatt Grading

In [11]:
def heckmatt(z):
    """Heckmatt et al. (1982) — Grade I=Normal, II=Mild, III=Moderate, IV=Severe"""
    if z<-0.5: return 1
    elif z<0.5: return 2
    elif z<1.5: return 3
    else: return 4

df['heckmatt']=df['ei_zscore'].apply(heckmatt)
gc=df['heckmatt'].value_counts().sort_index()
gl={1:'Grade I\n(Normal)',2:'Grade II\n(Mild)',3:'Grade III\n(Moderate)',4:'Grade IV\n(Severe)'}
print('Heckmatt Distribution:')
for g,c in gc.items(): print(f'  Grade {g}: {c:4d} ({c/len(df)*100:.1f}%)')
print('\nBy Side:')
print(df.groupby(['side','heckmatt']).size().unstack(fill_value=0))

colors=['#2ECC71','#F39C12','#E74C3C','#8E44AD']
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].bar([gl[g] for g in sorted(gc.index)],[gc[g] for g in sorted(gc.index)],color=colors[:len(gc)],edgecolor='k')
axes[0].set_title('Overall Heckmatt Distribution',fontsize=12); axes[0].set_ylabel('Count')
sg=df.groupby(['side','heckmatt']).size().unstack(fill_value=0)
sg.plot(kind='bar',ax=axes[1],color=colors[:4],edgecolor='k')
axes[1].set_title('Heckmatt by LM Side',fontsize=12); axes[1].tick_params(axis='x',rotation=0)
axes[1].legend(title='Grade',labels=[f'Grade {g}' for g in sg.columns])
plt.suptitle('Heckmatt Grading — LM Muscle EI (Fixed)',fontsize=14)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'heckmatt_grading.png'),dpi=100); plt.close()
print('Saved: heckmatt_grading.png')

Heckmatt Distribution:
  Grade 1:  128 (33.2%)
  Grade 2:  141 (36.5%)
  Grade 3:   86 (22.3%)
  Grade 4:   31 (8.0%)

By Side:
heckmatt   1   2   3   4
side                    
Center    44  59  29   8
Left      71  69  42  13
Right     13  13  15  10
Saved: heckmatt_grading.png


## Step 12: Left vs Right LM Analysis 


In [12]:
# Only dual-mask subjects for Left vs Right comparison
df_dual  = df[df['mask_type']=='dual'].copy()
left_ei  = df_dual[df_dual['side']=='Left']['ei_raw'].values
right_ei = df_dual[df_dual['side']=='Right']['ei_raw'].values

print(f'Dual-mask pairs — Left: n={len(left_ei)}, Right: n={len(right_ei)}')
if len(left_ei)>1 and len(right_ei)>1:
    stat,pval=mannwhitneyu(left_ei,right_ei,alternative='two-sided')
    print(f'Left  EI: {left_ei.mean():.4f} ± {left_ei.std():.4f}')
    print(f'Right EI: {right_ei.mean():.4f} ± {right_ei.std():.4f}')
    print(f'Mann-Whitney p={pval:.4f} ({"Significant" if pval<0.05 else "Not significant"})')
else:
    pval=1.0; print('Not enough data for comparison')

# Per-subject asymmetry
dl=df_dual[df_dual['side']=='Left'][['subject_id','visit_id','ei_raw','csa_cm2','ei_zscore']]
dl=dl.rename(columns={'ei_raw':'ei_l','csa_cm2':'csa_l','ei_zscore':'z_l'})
dr=df_dual[df_dual['side']=='Right'][['subject_id','visit_id','ei_raw','csa_cm2','ei_zscore']]
dr=dr.rename(columns={'ei_raw':'ei_r','csa_cm2':'csa_r','ei_zscore':'z_r'})
dsym=dl.merge(dr,on=['subject_id','visit_id'])
if len(dsym)>0:
    dsym['ei_asym_pct']  =abs(dsym['ei_l']-dsym['ei_r'])/((dsym['ei_l']+dsym['ei_r'])/2+1e-8)*100
    dsym['csa_asym_pct'] =abs(dsym['csa_l']-dsym['csa_r'])/((dsym['csa_l']+dsym['csa_r'])/2+1e-8)*100
    # Check for dominance pattern
    left_higher  = (dsym['ei_l'] > dsym['ei_r']).sum()
    right_higher = (dsym['ei_r'] > dsym['ei_l']).sum()
    print(f'\nPaired records : {len(dsym)}')
    print(f'EI asymmetry   : {dsym["ei_asym_pct"].mean():.2f}% ± {dsym["ei_asym_pct"].std():.2f}%')
    print(f'CSA asymmetry  : {dsym["csa_asym_pct"].mean():.2f}% ± {dsym["csa_asym_pct"].std():.2f}%')
    print(f'Left EI higher : {left_higher}/{len(dsym)} subjects')
    print(f'Right EI higher: {right_higher}/{len(dsym)} subjects')
    print(f'(0% asymmetry = identical, now should be >0 with fix)')

fig,axes=plt.subplots(1,3,figsize=(16,5))
bp=axes[0].boxplot([left_ei,right_ei],labels=['Left LM','Right LM'],patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue'); bp['boxes'][1].set_facecolor('lightcoral')
axes[0].set_title(f'EI: Left vs Right LM\n(p={pval:.4f})',fontsize=11); axes[0].set_ylabel('Raw EI')
lc=df_dual[df_dual['side']=='Left']['csa_cm2'].values
rc=df_dual[df_dual['side']=='Right']['csa_cm2'].values
bp2=axes[1].boxplot([lc,rc],labels=['Left LM','Right LM'],patch_artist=True)
bp2['boxes'][0].set_facecolor('lightblue'); bp2['boxes'][1].set_facecolor('lightcoral')
axes[1].set_title('CSA (cm²): Left vs Right LM',fontsize=11); axes[1].set_ylabel('CSA (cm²)')
if len(dsym)>0:
    axes[2].hist(dsym['ei_asym_pct'],bins=15,color='steelblue',edgecolor='k',alpha=0.8)
    axes[2].axvline(dsym['ei_asym_pct'].mean(),color='red',linestyle='--',
                    label=f'Mean={dsym["ei_asym_pct"].mean():.2f}%')
    axes[2].set_title('Real EI Asymmetry (%)',fontsize=11); axes[2].set_xlabel('%'); axes[2].legend()
plt.suptitle('Left vs Right LM — Dual-Mask Subjects (n=27) — FIXED',fontsize=13)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'side_analysis.png'),dpi=100); plt.close()
print('Saved: side_analysis.png')

Dual-mask pairs — Left: n=45, Right: n=45
Left  EI: 0.2381 ± 0.0756
Right EI: 0.2589 ± 0.0785
Mann-Whitney p=0.1726 (Not significant)

Paired records : 45
EI asymmetry   : 15.93% ± 15.64%
CSA asymmetry  : 20.29% ± 10.66%
Left EI higher : 14/45 subjects
Right EI higher: 31/45 subjects
(0% asymmetry = identical, now should be >0 with fix)
Saved: side_analysis.png


## Step 13: Bland-Altman Plot

In [13]:
m1=df['ei_raw'].values; m2=df['ei_phantom'].values/100
ba_mean=(m1+m2)/2; ba_diff=m1-m2
bd,bs=ba_diff.mean(),ba_diff.std(); bu=bd+1.96*bs; bl=bd-1.96*bs
plt.figure(figsize=(10,6))
plt.scatter(ba_mean,ba_diff,alpha=0.5,s=25,color='steelblue',edgecolors='k',linewidths=0.3)
plt.axhline(bd,color='red',linewidth=2,label=f'Mean diff={bd:.4f}')
plt.axhline(bu,color='orange',linestyle='--',linewidth=1.5,label=f'+1.96SD={bu:.4f}')
plt.axhline(bl,color='orange',linestyle='--',linewidth=1.5,label=f'-1.96SD={bl:.4f}')
plt.fill_between([ba_mean.min(),ba_mean.max()],bl,bu,alpha=0.08,color='orange')
plt.xlabel('Mean of Raw EI & Phantom-Std EI',fontsize=12)
plt.ylabel('Difference (Raw - Phantom-Std)',fontsize=12)
plt.title('Bland-Altman Plot\nLumbar Multifidus — LUMINOUS (Fixed)',fontsize=13)
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'bland_altman.png'),dpi=100); plt.close()
print(f'LoA: [{bl:.4f}, {bu:.4f}]'); print('Saved: bland_altman.png')

LoA: [-1.2253, -0.3244]
Saved: bland_altman.png


## Step 14: Longitudinal Reproducibility

In [14]:
ss=df.groupby('subject_id').agg(n_visits=('visit_id','count'),
    mean_ei=('ei_raw','mean'),std_ei=('ei_raw','std'),
    mean_csa=('csa_cm2','mean'),std_csa=('csa_cm2','std')).reset_index().fillna(0)
ss['cv_ei'] =(ss['std_ei'] /(ss['mean_ei'] +1e-8))*100
ss['cv_csa']=(ss['std_csa']/(ss['mean_csa']+1e-8))*100
multi_v=ss[ss['n_visits']>1]
print(f'Multi-visit subjects: {len(multi_v)}')
print(f'Mean within-subject EI  CV: {multi_v["cv_ei"].mean():.2f}%')
print(f'Mean within-subject CSA CV: {multi_v["cv_csa"].mean():.2f}%')
top_s=multi_v.nlargest(8,'n_visits')['subject_id'].tolist()

fig,axes=plt.subplots(2,2,figsize=(16,12))
for sid in top_s:
    sub=df[df['subject_id']==sid].sort_values('visit_int')
    axes[0][0].plot(sub['visit_int'],sub['ei_raw'],'o-',label=f'S{sid}',alpha=0.8)
    axes[0][1].plot(sub['visit_int'],sub['ei_zscore'],'o-',label=f'S{sid}',alpha=0.8)
    axes[1][0].plot(sub['visit_int'],sub['csa_cm2'],'o-',label=f'S{sid}',alpha=0.8)
for ax,title,ylabel in zip([axes[0][0],axes[0][1],axes[1][0]],
    ['Raw EI Across Sessions','Z-score EI Across Sessions','CSA (cm²) Across Sessions'],
    ['Raw EI','Z-score','CSA (cm²)']):
    ax.set_title(title); ax.set_xlabel('Session ID'); ax.set_ylabel(ylabel)
    ax.legend(fontsize=7,ncol=2)
if len(multi_v)>0:
    axes[1][1].hist(multi_v['cv_ei'].clip(0,50),bins=20,color='steelblue',edgecolor='k',alpha=0.8)
    axes[1][1].axvline(multi_v['cv_ei'].mean(),color='red',linestyle='--',
                       label=f'Mean={multi_v["cv_ei"].mean():.2f}%')
    axes[1][1].set_title('Within-Subject EI CV (Longitudinal Reproducibility)')
    axes[1][1].set_xlabel('CV (%)'); axes[1][1].legend()
plt.suptitle('Longitudinal LM EI & CSA — LUMINOUS Database (All Prone Rest)',fontsize=14)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'longitudinal_analysis.png'),dpi=100); plt.close()
print('Saved: longitudinal_analysis.png')

Multi-visit subjects: 107
Mean within-subject EI  CV: 13.48%
Mean within-subject CSA CV: 10.85%
Saved: longitudinal_analysis.png


## Step 15: Save All Results & Final Summary

In [15]:
df.to_csv(os.path.join(OUTPUT_DIR,'echointensity_results.csv'),index=False)
ss.to_csv(os.path.join(OUTPUT_DIR,'subject_stats.csv'),index=False)
if len(dsym)>0: dsym.to_csv(os.path.join(OUTPUT_DIR,'side_asymmetry.csv'),index=False)
pd.DataFrame({'epoch':list(range(1,n_ep+1)),'train_loss':trl,'val_loss':vll,
    'train_dice':trd,'val_dice':vld}).to_csv(os.path.join(OUTPUT_DIR,'training_history.csv'),index=False)

print('\n'+'='*65)
print('  FINAL SUMMARY — Improved LUMINOUS LM Segmentation')
print('  Dataset: Belasso et al. (2020), BMC Musculoskelet Disord')
print('='*65)
print(f'  Images    : 341 B-mode | 386 annotations | All PRONE REST')
print(f'  Subjects  : {df["subject_id"].nunique()} | Multi-visit: {len(multi_v)}')
print(f'  Strategy  : Midpoint crop for dual-mask (verified gap=20-73px)')
print()
print('  SEGMENTATION:')
print(f'    Dice: {md:.4f} | IoU: {mi:.4f} | Prec: {mp:.4f} | Recall: {mr:.4f}')
for s,dices in side_dice.items():
    if dices: print(f'    {s:6s} Dice: {np.mean(dices):.4f} (n={len(dices)})')
print()
print('  ECHOINTENSITY:')
print(f'    Mean EI (raw)    : {df["ei_raw"].mean():.4f} ± {df["ei_raw"].std():.4f}')
print(f'    Mean CSA         : {df["csa_cm2"].mean():.2f} cm²')
print(f'    CV (raw)         : {cv_raw:.2f}%')
print(f'    CV (phantom std) : {cv_phantom:.2f}%')
print(f'    Heckmatt         : {dict(df["heckmatt"].value_counts().sort_index())}')
print()
if len(dsym)>0:
    print(f'  SIDE ASYMMETRY (n={len(dsym)} pairs, REAL values now):')
    print(f'    EI asymmetry  : {dsym["ei_asym_pct"].mean():.2f}% (was 0% before fix)')
    print(f'    CSA asymmetry : {dsym["csa_asym_pct"].mean():.2f}%')
    print(f'    Left>Right EI : {(dsym["ei_l"]>dsym["ei_r"]).sum()}/{len(dsym)}')
print()
print(f'  LONGITUDINAL: Mean within-subject CV = {multi_v["cv_ei"].mean():.2f}%')
print()
print('  Output files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f'    {f:45s} {os.path.getsize(os.path.join(OUTPUT_DIR,f))/1024:.1f} KB')
print('='*65)


  FINAL SUMMARY — Improved LUMINOUS LM Segmentation
  Dataset: Belasso et al. (2020), BMC Musculoskelet Disord
  Images    : 341 B-mode | 386 annotations | All PRONE REST
  Subjects  : 109 | Multi-visit: 107
  Strategy  : Midpoint crop for dual-mask (verified gap=20-73px)

  SEGMENTATION:
    Dice: 0.9120 | IoU: 0.8434 | Prec: 0.9219 | Recall: 0.0720
    Left   Dice: 0.9138 (n=39)
    Right  Dice: 0.9100 (n=11)
    Center Dice: 0.9102 (n=28)

  ECHOINTENSITY:
    Mean EI (raw)    : 0.2252 ± 0.0669
    Mean CSA         : 4.65 cm²
    CV (raw)         : 29.66%
    CV (phantom std) : 29.70%
    Heckmatt         : {1: 128, 2: 141, 3: 86, 4: 31}

  SIDE ASYMMETRY (n=45 pairs, REAL values now):
    EI asymmetry  : 15.93% (was 0% before fix)
    CSA asymmetry : 20.29%
    Left>Right EI : 14/45

  LONGITUDINAL: Mean within-subject CV = 13.48%

  Output files:
    bland_altman.png                              61.9 KB
    echointensity_results.csv                     70.1 KB
    heckmatt_gradin